In [ ]:
!pip install sagemaker==2.224.1

In [ ]:
import  boto3
import pandas as pd 
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession
s3_client = boto3.resource('s3') 
pipeline_name = f"sagemaker-mlops-train-pipeline"
sagemaker_session = sagemaker.session.Session() 
region = sagemaker_session.boto_region_name 
role = sagemaker.get_execution_role() 
pipeline_session = PipelineSession() 
default_bucket = sagemaker_session.default_bucket()
model_package_group_name = f"BankModelPackageGroup"

In [ ]:
from sagemaker.workflow.parameters import (ParameterInteger, ParameterString, ParameterFloat)
auc_score_threshold = 0.75
base_job_prefix = "bank-example"
processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1) 
processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value="ml.t3.medium") 
training_instance_type = ParameterString(name = "TrainingInstanceType", default_value="ml.t3.medium") 
input_data = "bank-full.csv"
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")

In [ ]:
# !wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
# !unzip bank+marketing.zip

In [ ]:
# !unzip bank.zip

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

framework_version = "1.0-1"

sklearn_processor = SKLearnProcessor(
    framework_version=framework_version,
    instance_type="ml.t3.medium",
    instance_count=processing_instance_count,
    base_job_name="sklearn-bank-process",
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{default_bucket}/output/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation",   # ✅ FIXED
            destination=f"s3://{default_bucket}/output/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",         # ✅ FIXED
            destination=f"s3://{default_bucket}/output/test"
        )
    ],
    code="preprocess-bank.py",   # no f needed
)

step_process = ProcessingStep(
    name="BankModelProcess",
    step_args=processor_args
)

In [ ]:
import sagemaker
from sagemaker.estimator import Estimator

# =========================
# 1. Session + Role
# =========================
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

# =========================
# 2. Basic configs
# =========================
region = sagemaker_session.boto_region_name
default_bucket = sagemaker_session.default_bucket()

training_instance_type = "ml.m5.large"

# =========================
# 3. XGBoost container
# =========================
image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.0-1",
    py_version="py3",
    instance_type=training_instance_type,
)

# =========================
# 4. Output path (model artifacts)
# =========================
model_path = f"s3://{default_bucket}/output"

# =========================
# 5. Fixed hyperparameters (SAFE ones)
# =========================
fixed_hyperparameters = {
    "eval_metric": "auc",              # since you're using AUC
    "objective": "binary:logistic",    # binary classification
    "num_round": "100",
    "max_depth": "5",
    "eta": "0.2",
    "subsample": "0.8",
    "colsample_bytree": "0.8"
}

# =========================
# 6. Create Estimator (training job config)
# =========================
xgb_train = Estimator(
    image_uri=image_uri,
    instance_type=training_instance_type,
    instance_count=1,
    hyperparameters=fixed_hyperparameters,
    output_path=model_path,
    base_job_name="my-xgb-model",   # change name if needed
    sagemaker_session=sagemaker_session,
    role=role,
)

In [ ]:
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    HyperparameterTuner,
)
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TuningStep

# =========================
# 1. Hyperparameter ranges
# =========================
hyperparameter_ranges = {
    "eta": ContinuousParameter(0.01, 0.3),
    "min_child_weight": IntegerParameter(1, 10),   # ✅ FIXED
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
}

# =========================
# 2. Metric definition (VERY IMPORTANT)
# =========================
objective_metric_name = "validation:auc"

metric_definitions = [
    {
        "Name": "validation:auc",
        "Regex": "validation-auc:([0-9\\.]+)"
    }
]

# =========================
# 3. Tuner
# =========================
tuner = HyperparameterTuner(
    estimator=xgb_train,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=metric_definitions,   # ✅ REQUIRED
    max_jobs=2,
    max_parallel_jobs=2,
    objective_type="Maximize"
)

step_tuning = TuningStep(
    name="BankHyperParameterTuning",
    tuner=tuner,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)


In [ ]:
# define model evaluation step to evaluate the trained model
from sagemaker.processing import ScriptProcessor
script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="script-bank-eval",
    role=role,
    sagemaker_session=pipeline_session,
)
eval_args = script_eval.run(
     inputs=[
            ProcessingInput(
                source=step_tuning.get_top_model_s3_uri(top_k=0,s3_bucket=default_bucket,prefix="output"),
                destination="/opt/ml/processing/model"
            ),
            ProcessingInput(
                source=step_process.properties.ProcessingOutputConfig.Outputs[
                    "test"
                ].S3Output.S3Uri,
                destination="/opt/ml/processing/test"
            )
        ],
    outputs=[
            ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation",\
                             destination=f"s3://{default_bucket}/output/evaluation"),
        ],
    code=f"evaluation-bank.py",
)
from sagemaker.workflow.properties import PropertyFile

evaluation_report = PropertyFile(
    name="BankEvaluationReport", output_name="evaluation", path="evaluation.json"
)
step_eval = ProcessingStep(
    name="BankEvalModel",
    step_args=eval_args,
    property_files=[evaluation_report],
)

In [ ]:
from sagemaker import Model
from sagemaker.workflow.model_step import ModelStep

model = Model(
    image_uri=image_uri,
    model_data=step_tuning.get_top_model_s3_uri(top_k=0,s3_bucket=default_bucket,prefix="output"),
    sagemaker_session=pipeline_session,
    role=role,
)
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)
register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="BankRegisterModel", step_args=register_args)

In [ ]:
from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
cond_lte = ConditionGreaterThan(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.auc_score.value",
    ),
    right=auc_score_threshold,
)
step_cond = ConditionStep(
    name="CheckAUCScoreBankEvaluation",
    conditions=[cond_lte],
    if_steps=[step_register],
)

In [ ]:
import json
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        processing_instance_type,
        training_instance_type,
        model_approval_status,
        input_data,
        auc_score_threshold,
    ],
    steps=[step_process, step_tuning, step_eval, step_cond],
) 
definition = json.loads(pipeline.definition())
print(definition)

In [ ]:
# Create a new or update existing Pipeline
pipeline.upsert(role_arn=role)
# start Pipeline execution
pipeline.start()